# Notebook 4: Evaluating Generated Materials — MLIPs

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

In this notebook you will learn to:
- Understand why **evaluation** is critical after generation
- Use a **Machine Learning Interatomic Potential** (CHGNet) to evaluate structures
- Predict energies, forces, and stresses in seconds (vs. hours with DFT)
- Relax a crystal structure using an MLIP
- Connect evaluation to SCIGEN's screening pipeline

> **Prerequisites:** Run `00_setup.ipynb` first (CHGNet was installed there).

In [ ]:
# --- Run once per Colab session (skip if you already ran 00_setup.ipynb) ---
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

# Clone repo if needed
if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')

# Install all tutorial dependencies from the shared requirements file
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', f'{PROJECT_DIR}/requirements-colab.txt'])
os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')

---
## 4.1 The evaluation problem

Generating crystal structures is only half the job. After generation, we need to ask:

1. **Is this structure physically reasonable?** (sensible bond lengths, coordination)
2. **Is it thermodynamically stable?** (low energy, near the convex hull)
3. **Does it have interesting properties?** (magnetic, electronic, mechanical)

### The traditional approach: DFT
Density Functional Theory (DFT) gives accurate energies and forces, but it's
**expensive** — each structure takes minutes to hours on a supercomputer.
If you generated 10,000 candidates, DFT relaxation could take weeks.

### The modern approach: MLIPs
Machine Learning Interatomic Potentials are trained on DFT data and can predict
energies/forces/stresses in **seconds** — 1000x faster than DFT.
They enable rapid screening of large candidate sets.

---
## 4.2 CHGNet: a universal MLIP

[CHGNet](https://github.com/CederGroupHub/chgnet) (Crystal Hamiltonian Graph Neural Network)
is a universal potential trained on the Materials Project database.
It can predict energies, forces, stresses, and magnetic moments for any inorganic material.

Let's load the pretrained model.

In [ ]:
from chgnet.model.model import CHGNet

chgnet = CHGNet.load()
print(f'CHGNet loaded: {sum(p.numel() for p in chgnet.parameters()):,} parameters')

---
## 4.3 Predicting properties

Let's predict the energy of a known structure from the training data.

In [ ]:
import os
import pandas as pd
from pymatgen.core import Structure

PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')
df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'test.csv'))

# Pick a structure
row = df.iloc[0]
struct = Structure.from_str(row['cif'], fmt='cif')

print(f'Structure: {row["pretty_formula"]} ({row["material_id"]})')
print(f'Atoms: {struct.num_sites}')
print(f'DFT formation energy: {row["formation_energy_per_atom"]:.4f} eV/atom')

In [ ]:
# Predict with CHGNet
prediction = chgnet.predict_structure(struct)

print(f'CHGNet predictions for {row["pretty_formula"]}:')
print(f'  Energy:  {prediction["e"]:>10.4f} eV/atom')
print(f'  Forces:  max |F| = {abs(prediction["f"]).max():.4f} eV/Å')
print(f'  Stress:  {prediction["s"].diagonal().tolist()}')
if 'm' in prediction:
    print(f'  Magmom:  {prediction["m"].tolist()}')

---
## 4.4 Structure relaxation

Relaxation optimizes the atomic positions and lattice to minimize the energy.
This is like letting the structure "settle" into its lowest-energy configuration.

With CHGNet, this takes seconds instead of hours.

In [ ]:
from chgnet.model.dynamics import StructOptimizer

optimizer = StructOptimizer()

# Relax the structure
result = optimizer.relax(struct, verbose=True)

relaxed = result['final_structure']
print(f'\nRelaxation complete.')
print(f'  Initial energy: {prediction["e"]:.4f} eV/atom')
print(f'  Final energy:   {result["trajectory"].energies[-1] / relaxed.num_sites:.4f} eV/atom')

In [ ]:
# Compare original and relaxed structures
import numpy as np

print('Lattice parameter changes:')
for param in ['a', 'b', 'c', 'alpha', 'beta', 'gamma']:
    orig = getattr(struct.lattice, param)
    relax = getattr(relaxed.lattice, param)
    unit = 'Å' if param in ['a', 'b', 'c'] else '°'
    print(f'  {param:>5s}: {orig:8.3f} → {relax:8.3f} {unit}  (Δ = {relax - orig:+.3f})')

# Atomic displacement
displacements = np.linalg.norm(
    relaxed.cart_coords - struct.cart_coords, axis=1
)
print(f'\nMax atomic displacement: {displacements.max():.4f} Å')
print(f'Mean atomic displacement: {displacements.mean():.4f} Å')

### Relaxation trajectory

Let's visualize how the energy decreased during relaxation:

In [ ]:
import matplotlib.pyplot as plt

# Plot energy trajectory during relaxation
energies_traj = result['trajectory'].energies
n_atoms_relax = relaxed.num_sites

fig, ax = plt.subplots(figsize=(8, 4))
steps = range(len(energies_traj))
ax.plot(steps, [e / n_atoms_relax for e in energies_traj],
        'o-', color='steelblue', markersize=4, linewidth=1.5)
ax.set_xlabel('Optimization step', fontsize=12)
ax.set_ylabel('Energy (eV/atom)', fontsize=12)
ax.set_title('CHGNet relaxation trajectory', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Converged in {len(energies_traj)} steps')
print(f'Energy change: {(energies_traj[-1] - energies_traj[0]) / n_atoms_relax:.4f} eV/atom')

### Structure animation during relaxation

Let's visualize how the atomic positions change during relaxation.
We can create an animation showing the structure at key steps of the optimization,
revealing how the atoms "settle" into their equilibrium positions.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np

# Extract trajectory frames
traj = result['trajectory']
n_frames = len(traj.energies)

# Select key frames (evenly spaced, max 20 frames for smooth animation)
n_show = min(n_frames, 20)
frame_indices = np.linspace(0, n_frames - 1, n_show, dtype=int)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')

# Get coordinate range for consistent axis limits across frames
all_positions = np.array([traj.atom_positions[i] for i in frame_indices])
pad = 0.5
x_lim = (all_positions[:, :, 0].min() - pad, all_positions[:, :, 0].max() + pad)
y_lim = (all_positions[:, :, 1].min() - pad, all_positions[:, :, 1].max() + pad)
z_lim = (all_positions[:, :, 2].min() - pad, all_positions[:, :, 2].max() + pad)

species_list = [str(s.specie) for s in struct]
unique_el = sorted(set(species_list))
cmap_colors = plt.cm.tab10(np.linspace(0, 0.9, max(len(unique_el), 1)))
color_map = {el: cmap_colors[i] for i, el in enumerate(unique_el)}

def update(frame_num):
    ax.clear()
    idx = frame_indices[frame_num]
    positions = traj.atom_positions[idx]
    energy = traj.energies[idx] / struct.num_sites

    for el in unique_el:
        mask = [s == el for s in species_list]
        coords = positions[mask]
        ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2],
                   s=200, c=[color_map[el]], label=el,
                   alpha=0.85, edgecolors='k', linewidths=0.5)

    ax.set_xlim(*x_lim)
    ax.set_ylim(*y_lim)
    ax.set_zlim(*z_lim)
    ax.set_xlabel('x (Å)')
    ax.set_ylabel('y (Å)')
    ax.set_zlabel('z (Å)')
    ax.legend(loc='upper left', fontsize=8)
    ax.set_title(f'Step {idx}/{n_frames-1}  |  E = {energy:.4f} eV/atom', fontsize=11)

anim = FuncAnimation(fig, update, frames=n_show, interval=300, repeat=True)
plt.close(fig)

try:
    display(HTML(anim.to_jshtml()))
except Exception:
    # Fallback: show first, middle, and last frames as static plots
    print('Animation display not supported — showing key frames instead.')
    fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4), subplot_kw={'projection': '3d'})
    for ax2, label, fi in zip(axes2, ['Initial', 'Mid', 'Final'],
                               [0, n_show // 2, n_show - 1]):
        update.__wrapped__ = None  # reset
        idx = frame_indices[fi]
        positions = traj.atom_positions[idx]
        energy = traj.energies[idx] / struct.num_sites
        for el in unique_el:
            mask = [s == el for s in species_list]
            coords = positions[mask]
            ax2.scatter(coords[:, 0], coords[:, 1], coords[:, 2],
                       s=200, c=[color_map[el]], label=el,
                       alpha=0.85, edgecolors='k', linewidths=0.5)
        ax2.set_xlim(*x_lim); ax2.set_ylim(*y_lim); ax2.set_zlim(*z_lim)
        ax2.set_xlabel('x (Å)'); ax2.set_ylabel('y (Å)'); ax2.set_zlabel('z (Å)')
        ax2.set_title(f'{label} (step {idx}, E={energy:.4f})', fontsize=10)
        ax2.legend(loc='upper left', fontsize=7)
    plt.tight_layout()
    plt.show()

In [ ]:
# Atomic displacement analysis: how much did each atom move?
initial_pos = traj.atom_positions[0]
final_pos = traj.atom_positions[-1]
displacements = np.linalg.norm(final_pos - initial_pos, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Per-atom displacement bar chart
colors_per_atom = [color_map[el] for el in species_list]
axes[0].bar(range(len(displacements)), displacements, color=colors_per_atom, edgecolor='white')
axes[0].set_xlabel('Atom index', fontsize=11)
axes[0].set_ylabel('Displacement (Å)', fontsize=11)
axes[0].set_title('Atomic displacements during relaxation', fontsize=12)

# Add element labels
for i, el in enumerate(species_list):
    axes[0].text(i, displacements[i] + 0.002, el, ha='center', fontsize=7, color='gray')

# Displacement histogram
axes[1].hist(displacements, bins=15, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Displacement (Å)', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('Distribution of atomic displacements', fontsize=12)
axes[1].axvline(displacements.mean(), color='red', linestyle='--',
                label=f'Mean = {displacements.mean():.4f} Å')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Most displaced atom: {species_list[displacements.argmax()]} '
      f'(index {displacements.argmax()}, {displacements.max():.4f} Å)')
print(f'Least displaced atom: {species_list[displacements.argmin()]} '
      f'(index {displacements.argmin()}, {displacements.min():.4f} Å)')

---
## 4.5 Batch evaluation

One of the key advantages of MLIPs is speed. Let's evaluate many structures
and compare CHGNet formation energies against DFT values.

In [ ]:
import matplotlib.pyplot as plt
import time
import numpy as np

# Evaluate the first 20 structures in the test set
n_eval = min(20, len(df))
energies_chgnet = []
energies_dft = []
formulas = []

# Compute CHGNet elemental reference energies for formation energy calculation
# (energy of each element in its standard bulk form)
from pymatgen.core import Structure, Lattice, Element

def get_elemental_ref_energies(chgnet_model):
    """Get CHGNet total energy per atom for common elemental reference structures."""
    refs = {}
    # Use simple structures as references
    # In practice, the MP elemental references are more accurate,
    # but this gives a reasonable approximation for comparison
    for el_symbol in ['Li', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ti', 'V', 'Cr',
                      'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se',
                      'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Ru', 'Rh', 'Pd', 'Ag',
                      'Cd', 'In', 'Sn', 'Sb', 'Te', 'Cs', 'Ba', 'La', 'Hf', 'Ta',
                      'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Pb', 'Bi',
                      'O', 'S', 'N', 'F', 'Cl', 'Br', 'I', 'P', 'C', 'B', 'H']:
        try:
            el = Element(el_symbol)
            # BCC structure as rough reference
            lat = Lattice.cubic(3.0)
            s = Structure(lat, [el_symbol, el_symbol], [[0,0,0], [0.5,0.5,0.5]])
            pred = chgnet_model.predict_structure(s)
            refs[el_symbol] = float(pred['e'])
        except:
            pass
    return refs

print('Computing elemental reference energies...')
elem_refs = get_elemental_ref_energies(chgnet)
print(f'  References computed for {len(elem_refs)} elements')

start = time.time()
for i in range(n_eval):
    row = df.iloc[i]
    try:
        s = Structure.from_str(row['cif'], fmt='cif')
        pred = chgnet.predict_structure(s)

        # Compute formation energy: E_total - sum(x_i * E_ref_i)
        total_e = float(pred['e'])  # eV/atom
        comp = s.composition.fractional_composition
        ref_sum = sum(comp[el] * elem_refs.get(str(el), 0) for el in comp)
        form_e = total_e - ref_sum

        energies_chgnet.append(form_e)
        energies_dft.append(row['formation_energy_per_atom'])
        formulas.append(row['pretty_formula'])
    except Exception as e:
        print(f'  Skipping {row["pretty_formula"]}: {e}')

elapsed = time.time() - start
print(f'Evaluated {len(energies_chgnet)} structures in {elapsed:.1f}s')
print(f'({elapsed/len(energies_chgnet)*1000:.0f} ms per structure)')

### Speed comparison: DFT vs. MLIP

One of the key advantages of MLIPs — they're orders of magnitude faster than DFT:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))

methods = ['DFT\n(VASP/QE)', 'CHGNet\n(MLIP)']
times = [3600, elapsed / len(energies_chgnet)]  # seconds per structure
colors = ['#E69F00', 'steelblue']

bars = ax.barh(methods, times, color=colors, edgecolor='white', height=0.5)
ax.set_xscale('log')
ax.set_xlabel('Time per structure (seconds)', fontsize=11)
ax.set_title('DFT vs. MLIP evaluation speed', fontsize=13)

# Annotate bars
for bar, t in zip(bars, times):
    if t >= 60:
        label = f'{t/60:.0f} min'
    else:
        label = f'{t:.2f} s'
    ax.text(bar.get_width() * 1.5, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0.001, 50000)
plt.tight_layout()
plt.show()

speedup = 3600 / (elapsed / len(energies_chgnet))
print(f'CHGNet is ~{speedup:.0f}x faster than DFT for energy evaluation')

In [ ]:
if not energies_dft:
    print('No data — run the batch evaluation cell above first.')
else:
    fig, ax = plt.subplots(figsize=(6, 6))

    ax.scatter(energies_dft, energies_chgnet, s=50, c='steelblue',
               edgecolors='white', linewidths=0.5)

    # Diagonal line
    all_vals = energies_dft + energies_chgnet
    lims = [min(all_vals) - 0.2, max(all_vals) + 0.2]
    ax.plot(lims, lims, 'k--', alpha=0.3, label='Perfect agreement')

    # Compute MAE
    mae = np.mean(np.abs(np.array(energies_dft) - np.array(energies_chgnet)))

    ax.set_xlabel('DFT formation energy (eV/atom)', fontsize=12)
    ax.set_ylabel('CHGNet formation energy (eV/atom)', fontsize=12)
    ax.set_title(f'CHGNet vs. DFT — Formation Energy\nMAE = {mae:.3f} eV/atom', fontsize=13)
    ax.legend()
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()

    print(f'Note: CHGNet elemental references are approximate (BCC structures).')
    print(f'For production use, use MP-computed elemental reference energies.')

---
## 4.6 Connection to SCIGEN's screening pipeline

SCIGEN includes its own GNN-based screening models (in `gnn_eval/`) that evaluate:
- **Stability** — Is the structure near the convex hull?
- **Magnetism** — Does the structure have interesting magnetic properties?

The full SCIGEN pipeline:
1. **Generate** — Create candidates with structural constraints (Notebook 05)
2. **Pre-screen** — Filter by composition validity, bond distances (`script/eval_screen.py`)
3. **MLIP screen** — Evaluate with GNN models or CHGNet
4. **DFT validation** — Final verification of the best candidates

In the published work, this pipeline narrowed 10 million generated structures
down to 24,743 DFT-validated candidates.

---
## Exercise

1. Pick a different structure from the test set. Relax it with CHGNet. How much did the energy change? How much did the atoms move?
2. Try relaxing the **NaCl structure** from Notebook 01 (build it from scratch). Does CHGNet predict the correct lattice parameter?

In [ ]:
# Your code here


---
## Key takeaways

1. **Generation without evaluation is useless.** You need fast, reliable ways to screen candidates.
2. **MLIPs (like CHGNet) are 1000x faster than DFT** and accurate enough for initial screening.
3. **Relaxation** lets you check if a generated structure is near an energy minimum.
4. **SCIGEN's pipeline** combines generation → pre-screening → MLIP evaluation → DFT validation.

In the next notebook, we'll generate crystal structures with SCIGEN!

---
## What's next?

Proceed to **Notebook 05: SCIGEN Generation** — the capstone demo where we generate crystal structures with targeted lattice constraints.